# Tutorial 90 — Retrieve through code relationships

Semantic and lexical retrieval treat chunks mainly as text. This tutorial uses
source structure itself by building a deterministic **code graph** from
the natural units introduced in [Tutorial 50](./50_source_structure.ipynb).

The graph represents code entities and relationships such as containment,
calls, imports, and inheritance. You will inspect neighboring entities, resolve
an explicitly named symbol, and retrieve directly connected code without asking
an LLM to guess which entity was intended.

[Tutorial 100](./100_persistent_indexes.ipynb) will persist the semantic,
lexical, and code-aware retrieval structures so they can be reused across
sessions.


## 1. Prepare QAChat

Each tutorial is independently runnable. These cells locate the repository,
make the local package importable, and call `prepare_notebook()` to verify the
project `.venv` environment. If Jupyter asks you to select the printed `.venv`
kernel, select it and rerun the cell.


In [ ]:
from pathlib import Path

root = Path.cwd()
if not (root / "qachat").is_dir():
    root = root.parent

print(f"QAChat root: {root}")


In [ ]:
import sys

if str(root) not in sys.path:
    sys.path.insert(0, str(root))


In [ ]:
from setup.notebook import prepare_notebook

root = prepare_notebook(root)


In [ ]:
import qachat


## 2. Prepare structured source units

The code graph reuses the natural source units introduced in
[Tutorial 50](./50_source_structure.ipynb) rather than rediscovering functions
and classes from arbitrary chunks. This preserves a single structural
representation across chunking and code-aware retrieval.

The simple word counter below is used only to bound very large units in this
model-independent example. Embedding-based indexing uses the embedding model's
tokenizer instead.


In [ ]:
report = qachat.scan_repository(
    root,
    extensions=[".py", ".m"],
    include_paths=["qachat"],
)

print(f"Selected {len(report['selected_files'])} files:")

for i, file in enumerate(report["selected_files"]):
    print(f"{i}. {file}")


In [ ]:
source_files = qachat.read_repository_files(report)
units = qachat.extract_source_units(source_files)

print(f"Read {len(source_files)} files")
print(f"Natural units: {len(units)}")


In [ ]:
def count_words(text: str) -> int:
    return len(text.split())


In [ ]:
chunks = qachat.chunk_source_units(
    units,
    count_tokens=count_words,
    target_tokens=300,
    max_tokens=400,
    overlap_tokens=40,
)

print(f"Searchable chunks: {len(chunks)}")


## 3. Build a code graph

The graph contains code entities and explicit relationships extracted from
source structure. Language-specific extractors can contribute relationships
such as calls, imports, containment, and inheritance without relying on folder
names or an LLM interpretation.


In [ ]:
code_graph = qachat.build_code_graph(source_files, units)

print(f"Entities: {len(code_graph.entities)}")
print(f"Relationships: {len(code_graph.relations)}")


In [ ]:
for relation in code_graph.relations:
    print(
        f"{relation.source} "
        f"--{relation.relation}--> "
        f"{relation.target}"
    )


You can choose one specific entity and inspect its immediate relationships:

- **Incoming** relationships show which code points to or contains the entity.
- **Outgoing** relationships show which code the entity points to or uses.

Here we inspect `qachat.chat._check_temperature`.

In [ ]:
target = "qachat.chat._check_temperature"

print(target)


In [ ]:
print("Incoming:")
for relation in code_graph.relations:
    if relation.target == target:
        print(
            f"  {relation.source} "
            f"--{relation.relation}--> {target}"
        )


In [ ]:
print("Outgoing:")
for relation in code_graph.relations:
    if relation.source == target:
        print(
            f"  {target} "
            f"--{relation.relation}--> {relation.target}"
        )


## 4. Resolve an explicit symbol

**Symbol resolution** is deterministic: the question must identify a code
entity that exists in the graph. QAChat does not ask an LLM to invent or guess which
function the user intended.


In [ ]:
QUESTION = "Which code is related to semantic_search?"

matches = qachat.resolve_code_entities(code_graph, QUESTION)

for match in matches:
    print(f"{match.kind}: {match.name}")


## 5. Retrieve code relationships

Distance 0 is the resolved entity itself. Distance 1 contains directly related
entities, for example callers, callees, imports, inherited structures, or
containment relationships represented by the graph. This channel therefore adds
structural evidence that text similarity alone may miss.


In [ ]:
results = qachat.code_search(
    code_graph,
    chunks,
    QUESTION,
    count=5,
    max_distance=1,
)

for rank, result in enumerate(results, start=1):
    print(
        f"{rank}. distance={result.distance} "
        f"symbol={result.chunk.symbol}"
    )


In [ ]:
for result in results:
    print(f"\n>>> {result.entity} <<<\n")
    print(result.chunk.text)


## Continue the tutorial series

**Previous:** [Tutorial 80 — Retrieve by lexical similarity](./80_lexical_retrieval.ipynb)

**Next:** [Tutorial 100 — Persist retrieval indexes](./100_persistent_indexes.ipynb)
